User Managemnet API

In [ ]:
from typing import List, Optional, Dict
from dataclasses import dataclass
from datetime import datetime

@dataclass
class User:
    """User data class."""
    id: int
    username: str
    email: str
    created_at: datetime
    is_active: bool = True

class UserDatabase:
    """In-memory user database."""

    def __init__(self):
        self._users: Dict[int, User] = {}
        self._next_id: int = 1

    def create_user(self, username: str, email: str) -> User:
        """Create a new user."""
        user = User(
            id=self._next_id,
            username=username,
            email=email,
            created_at=datetime.now()
        )
        self._users[user.id] = user
        self._next_id += 1
        return user

    def get_user(self, user_id: int) -> Optional[User]:
        """Get user by ID."""
        return self._users.get(user_id)

    def list_users(self, active_only: bool = True) -> List[User]:
        """List all users."""
        users = list(self._users.values())
        if active_only:
            users = [u for u in users if u.is_active]
        return users

    def update_user(self, user_id: int, **kwargs) -> Optional[User]:
        """Update user fields."""
        user = self.get_user(user_id)
        if not user:
            return None

        for key, value in kwargs.items():
            if hasattr(user, key):
                setattr(user, key, value)

        return user

    def delete_user(self, user_id: int) -> bool:
        """Soft delete a user."""
        user = self.get_user(user_id)
        if not user:
            return False

        user.is_active = False
        return True

# Usage
db = UserDatabase()

# Create users
user1 = db.create_user("john_doe", "john@example.com")
user2 = db.create_user("jane_doe", "jane@example.com")

# Get user
user = db.get_user(1)
print(user)

# List users
users = db.list_users()
print(users)

# Update user
db.update_user(1, email="newemail@example.com")

# Delete user
db.delete_user(2)
print(db.list_users())  # Only user1 remains

Cache with Decorators

In [ ]:
from typing import Callable, Any, Dict, Tuple
from functools import wraps
import time

class Cache:
    """Centralized cache."""

    def __init__(self, ttl: int = 300):
        self.ttl = ttl  # Time to live in seconds
        self._cache: Dict[Tuple, Tuple[Any, float]] = {}

    def get(self, key: Tuple) -> Optional[Any]:
        """Get value from cache."""
        if key in self._cache:
            value, timestamp = self._cache[key]
            if time.time() - timestamp < self.ttl:
                return value
            else:
                del self._cache[key]  # Expired
        return None

    def set(self, key: Tuple, value: Any) -> None:
        """Set value in cache."""
        self._cache[key] = (value, time.time())

    def clear(self) -> None:
        """Clear all cache."""
        self._cache.clear()

# Global cache instance
cache = Cache(ttl=60)

def cached(func: Callable) -> Callable:
    """Decorator to cache function results."""

    @wraps(func)
    def wrapper(*args, **kwargs):
        # Create cache key from function name and arguments
        cache_key = (func.__name__, args, tuple(sorted(kwargs.items())))

        # Check cache
        cached_result = cache.get(cache_key)
        if cached_result is not None:
            print(f"Cache hit for {func.__name__}{args}")
            return cached_result

        # Compute result
        print(f"Cache miss for {func.__name__}{args}")
        result = func(*args, **kwargs)

        # Store in cache
        cache.set(cache_key, result)

        return result

    return wrapper

@cached
def expensive_computation(x: int, y: int) -> int:
    """Simulate expensive computation."""
    time.sleep(2)  # Simulate delay
    return x + y

# First call: slow (2 seconds)
result = expensive_computation(5, 10)
print(result)

# Second call: fast (cached)
result = expensive_computation(5, 10)
print(result)